##  IMporting Libraries and Dataframe

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path


def find_data_file(name, max_levels_up=4):
    """Locate a data file, walking up from the working directory.

    Lets the notebook run from the project folder or any subfolder of it.
    """
    start = Path.cwd().resolve()
    for folder in [start, *list(start.parents)[:max_levels_up]]:
        if (folder / name).is_file():
            return folder
    raise FileNotFoundError(
        f'Could not find {name!r}. Looked in {start} '
        f'and {max_levels_up} parent folders.'
    )


DATA_DIR = find_data_file('spam.csv')
SPAM_CSV = DATA_DIR / 'spam.csv'

In [ ]:
df = pd.read_csv(SPAM_CSV, encoding='latin1')

In [ ]:
df.sample(5)

In [ ]:
df.info()  #dataset info

In [ ]:
df.drop(columns=['Unnamed: 2','Unnamed: 3','Unnamed: 4'],inplace=True)

In [ ]:
df.head()

In [ ]:
df.shape  #shape of data

## Data preprocessing

In [ ]:
df.rename(columns={'v1':'output','v2':'text'},inplace=True)
df.head()

### data encoding

In [ ]:
from sklearn.preprocessing import LabelEncoder
encoder=LabelEncoder()

In [ ]:
df['output']=encoder.fit_transform(df['output'])   #making label encoders of output
df.head(3)

In [ ]:
df.isna().sum()

In [ ]:
df.duplicated().sum()  #checking for duplicates

In [ ]:
#removing duplicates
df=df.drop_duplicates(keep='first')

In [ ]:
df.head(3)  
#database after removing duplicates messages

In [ ]:
# checking no of hams and spams msgs
df['output'].value_counts()
# 0: hams
#1: spams


### Data visualization

In [ ]:
import matplotlib.pyplot as plt
plt.pie(df['output'].value_counts(),labels=['ham','spam'],autopct="%0.2f")  #percentage of ham and spam msgs
plt.show()

In [ ]:
# Data is imbalance


In [ ]:
import nltk
nltk.download('punkt_tab')


### Counting no of words and characters into sentences

In [ ]:
df['num_characters']=df['text'].apply(len)   #defines no of characters into msg
df.head()

In [ ]:
#num of words into sentence
df['num_words']=df['text'].apply(lambda x:len(nltk.word_tokenize(x)))  #counting words into a sentence
  

In [ ]:
df.head()

In [ ]:
#it breaks sentence into parts on the basis of sentences
df['num_sent']=df['text'].apply(lambda x:len(nltk.sent_tokenize(x)))  #counting words into a sentence


In [ ]:
df.head()

In [ ]:
df[['num_characters','num_words','num_sent']].describe()

In [ ]:
df[['num_characters','num_words','num_sent']].info()

In [ ]:
df[df['output'] == 0][['num_characters','num_words','num_sent']].describe()  # to check characteristics of ham msgs

In [ ]:
df[df['output'] == 1][['num_characters','num_words','num_sent']].describe()  # to check characteristics of spam msgs

### Data Visualization using histogram using his, paiplot and pandas

In [ ]:
# histogram
import seaborn as sns
sns.histplot(df[df['output']== 0]['num_characters'])
sns.histplot(df[df['output']== 1]['num_characters'])
plt.show()

In [ ]:


sns.countplot(x='output', data=df, color='green')
plt.title('Category Distribution')
plt.show()


In [ ]:
sns.pairplot(df,hue='output')
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

#heat map of numerical data values
numeric_df = df.select_dtypes(include=['int64', 'float64'])
sns.heatmap(numeric_df.corr(), annot=True)
plt.show()


### Data preprocessing 

In [ ]:
import nltk
nltk.download('stopwords')


In [ ]:

from nltk.corpus import stopwords
stopwords.words('english')   #stopwords
import string


In [ ]:
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()

In [ ]:
def transform_text(text):
    text=text.lower()   #converts text into lower case
    text=nltk.word_tokenize(text)   #splits text into words

    y=[]
    for i in text:
        if i.isalnum ():   #removing special characters
         y.append(i)

    text=y[:]  #copying list here 
    y.clear()

    for i in text:
        if i not in stopwords.words('english')and i not in string.punctuation:
         y.append(i)

    text=y[:]
    y.clear()
    
    for i in text:
      y.append(ps.stem(i))   #word stemming
    
    return  " ".join(y) 
   

In [ ]:
transform_text('hi its me alarming point here?')

In [ ]:
df['transformed text']=df['text'].apply(transform_text)

In [ ]:
df.head(3)

### Word cloud of spam and ham  msgs
## Ham word cloud

In [ ]:
from wordcloud import WordCloud

wc = WordCloud(
    width=500,
    height=500,
    min_font_size=10,
    background_color='white'   #spam messages wordcloud
)


In [ ]:
spam_wc=wc.generate(df[df['output']==1]['transformed text'].str.cat(sep=" "))

In [ ]:
plt.imshow(spam_wc)

## Ham word cloud

In [ ]:
Ham_wc=wc.generate(df[df['output']==0]['transformed text'].str.cat(sep=" "))
plt.imshow(Ham_wc)

## Finding mostly used  keywords in spam messages

In [ ]:
spam_corpus=[]
for msg in df[df['output']==1]['transformed text'].tolist():
    for word in msg.split():
        spam_corpus.append(word)

In [ ]:
len(spam_corpus)

In [ ]:
from collections import Counter
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Count top 35 words in spam corpus
word_count = pd.DataFrame(Counter(spam_corpus).most_common(35))  #count occuring of spam msgs and then convert it into dataframe
word_count.columns = ['word', 'count']  

# Plot
plt.figure(figsize=(10,6))
sns.barplot(x='count', y='word', data=word_count)
plt.xticks(rotation='horizontal')
plt.show()


### Mostly ised keywords in Ham messages

In [ ]:
Ham_corpus=[]
for msg in df[df['output']==0]['transformed text'].tolist():
    for word in msg.split():
        Ham_corpus.append(word)

In [ ]:
len(Ham_corpus)

In [ ]:

# Count top 35 words in spam corpus
word_count = pd.DataFrame(Counter(Ham_corpus).most_common(35))  #count occuring of spam msgs and then convert it into dataframe
word_count.columns = ['word', 'count']  

# Plot
plt.figure(figsize=(10,6))
sns.barplot(x='count', y='word', data=word_count)
plt.xticks(rotation='horizontal')
plt.show()

## Model Building and training Using Naive Bayes

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer,TfidfVectorizer
cv = CountVectorizer()
tfidf = TfidfVectorizer(max_features=3000)

In [ ]:
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB
from sklearn.metrics import accuracy_score,confusion_matrix,precision_score

In [ ]:
gnb=GaussianNB()
Mnb=MultinomialNB()
Bnb=BernoulliNB()

## Model training using tfidf vectorizer with multi Naive bayes models

## 1. Guassian naive bayes model 

In [ ]:
X_2 = tfidf.fit_transform(df['transformed text']).toarray()
y_2=df['output'].values

In [ ]:
X_2.shape

In [ ]:
y_2.shape

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(X_2, y_2,test_size=0.2, random_state=42)

In [ ]:
gnb.fit(X_train, y_train)
y2_pred1=gnb.predict(X_test)
print(f"Accuracy : {accuracy_score(y_test,y2_pred1)}")
print(f"Conusion matrix:\n {confusion_matrix(y_test,y2_pred1)}")
print(f"precision score:{precision_score(y_test,y2_pred1)}")

## 2. Multinomial Naive bayes model by tfidf

In [ ]:
Mnb.fit(X_train, y_train)
y2_pred2=Mnb.predict(X_test)
print(f"Accuracy : {accuracy_score(y_test,y2_pred2)}")
print(f"Conusion matrix:\n {confusion_matrix(y_test,y2_pred2)}")
print(f"precision score:{precision_score(y_test,y2_pred2)}")

## 3. Bernoulli Naive Bayes model by tfidf

In [ ]:
Bnb.fit(X_train, y_train)
y2_pred3=Bnb.predict(X_test)
print(f"Accuracy : {accuracy_score(y_test,y2_pred3)}")
print(f"Conusion matrix:\n {confusion_matrix(y_test,y2_pred3)}")
print(f"precision score:{precision_score(y_test,y2_pred3)}")

## Multimodels using here

Logistic Regression

SVc

Random Forest

Decision Tree

XGBoost 

KNN Classifier

GradientBooster

Multinomial Naive Bayes

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.ensemble import GradientBoostingClassifier
from xgboost import XGBClassifier

In [ ]:
svc = SVC(kernel='sigmoid', gamma=1.0)
knc = KNeighborsClassifier()
mnb = MultinomialNB()
dtc = DecisionTreeClassifier(max_depth=5)
lrc = LogisticRegression(solver='liblinear', penalty='l1')
rfc = RandomForestClassifier(n_estimators=50, random_state=2)
bc = BaggingClassifier(n_estimators=50, random_state=2)
etc = ExtraTreesClassifier(n_estimators=50, random_state=2)
gbdt = GradientBoostingClassifier(n_estimators=50,random_state=2)
xgb = XGBClassifier(n_estimators=50,random_state=2)

In [ ]:
clfs = {
    'SVC' : svc,
    'KN' : knc, 
    'NB': mnb, 
    'DT': dtc, 
    'LR': lrc, 
    'RF': rfc, 
    #'BgC': bc, 
    'ETC': etc,
    'GBDT':gbdt,
    'xgb':xgb
}

In [ ]:
def train_classifier(clf,X_train,y_train,X_test,y_test):
    clf.fit(X_train,y_train)
    y_pred = clf.predict(X_test)
    accuracy = accuracy_score(y_test,y_pred)
    precision = precision_score(y_test,y_pred)
    
    return accuracy,precision

In [ ]:
accuracy_scores = []
precision_scores = []

for name,clf in clfs.items():
    
    current_accuracy,current_precision = train_classifier(clf, X_train,y_train,X_test,y_test)
    
    print("For ",name)
    print("Accuracy - ",current_accuracy)
    print("Precision - ",current_precision)
    
    accuracy_scores.append(current_accuracy)
    precision_scores.append(current_precision)

In [ ]:
performance_df = pd.DataFrame({'Algorithm':clfs.keys(),'Accuracy':accuracy_scores,'Precision':precision_scores}).sort_values('Precision',ascending=False)

In [ ]:
performance_df 

In [ ]:
# Applying stacking
estimators=[('svm', svc), ('nb', mnb), ('et', etc)]
final_estimator=RandomForestClassifier()

In [ ]:
from sklearn.ensemble import StackingClassifier

In [ ]:
clf = StackingClassifier(estimators=estimators, final_estimator=final_estimator)

In [ ]:
clf.fit(X_train,y_train)
y_pred = clf.predict(X_test)
print("Accuracy",accuracy_score(y_test,y_pred))
print("Precision",precision_score(y_test,y_pred))


In [ ]:
import pickle

# Save the trained artefacts alongside the notebook.
with open(DATA_DIR / 'vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf, f)
with open(DATA_DIR / 'model.pkl', 'wb') as f:
    pickle.dump(Mnb, f)